# LM Studio
[[site]](https://lmstudio.ai/)

LM Studio - это Desktop приложение со своим UI для локального запуска LLM моделей<br>
Приложение написано поверх рантайма llama.cpp

Есть и SKD

Сервинг

OpenAI совместимый API = тот же входной и выходной формат, что для ChatGPT<br>Это сделано для универсальности и совместимости приложений. Сейчас по факту стандарт, все используют

С 2025 года в LM Studio можно подключать свои MCP-сервера

```python
import lmstudio as lms

model = lms.llm("qwen/qwen3-4b-2507")
result = model.respond("What is the meaning of life?")

print(result)
```

```
sdfsdf
```

# Ollama
[[site]](https://ollama.com)

Ollama - это прилодение и SDK (Python библиотека с клиентом) для локального запуска LLM моделей
Приложение написано поверх рантайма llama.cpp

# llama.cpp
[[site]](https://github.com/ggml-org/llama.cpp)

Низеоуровневая библиотека, которую в 2023 написал Болгарский разработчик для инференса LLAMA моделей. Она понравилась, её начало развивать комьюнити и туда добавили другие модели - главное, чтобы они были в формате GGUF

Основана на своем собсвтенном движке тензорных вычислений ggml. Это компактная альтернатива cuBLAS / PyTorch, но поскольку очень узкая (чисто под LLM forward pass), ее легко написали

Написана на C++, отсюда название

# vLLM
[[site]](https://github.com/vllm-project/vllm)

Инструмент сервинга, разрботанный в Berkley в 2023 году. Используется как production версия для крупных проектах<br>
Алгоритмическая новация - это PagedAttention

Как в vLLM работает serving:
- запускается API сервер, который принимает запросы совместимые с OpenAI стилизованном API
- использует HF TRansformers для загрузки модели
- подгружает свои CUDA ядра для PagedAttention
- параллельно с HTTP-сервером запускается Scheduler
    - ставит запросы в очередь
    - создает под них таски
    - проверяет статус выполнения
    - дергает ядра генерации
- если пользователь просил stream=True, то после каждой итерации отправляется новый токен 

Почему кастомные CUDA ядра?<br>
PyTorch ядра неоптимизированы под конкретную задачу. Утверждается, что можно ускорить обработку 1.5-3x

### PagedAttetnnion
[[paper]](https://arxiv.org/pdf/2309.06180)

__Идея:__ делить KV-Cache на части

## GGUF
Формат хранения весов модели

<img src="img/gguf.png" width=300>

Что содержит GGUF файл:
- словарь токенизатора
- веса слоев Attetnion (K, V, W, O), MLP (W1, W2)
- метаинформация (архитектура модели, параметры квантования(




# пример 1

In [ ]:
from vllm import LLM, SamplingParams

# 1. Создаём параметры сэмплинга
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=64,
)

# 2. Загружаем модель (любую HF)
llm = LLM(model="meta-llama/Llama-3-8B-Instruct")

# 3. Прогоняем запросы (можно список)
prompts = [
    "Explain the concept of Mixture-of-Experts in simple terms.",
    "Write a short haiku about GPUs.",
]

outputs = llm.generate(prompts, sampling_params)

# 4. Разбираем ответы
for i, out in enumerate(outputs):
    prompt = prompts[i]
    generated_text = out.outputs[0].text  # берём 0-й вариант
    print("PROMPT:", prompt)
    print("RESPONSE:", generated_text)
    print("=" * 80)


In [ ]:
Можно стримить ответ

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model="meta-llama/Llama-3-8B-Instruct")
sampling_params = SamplingParams(temperature=0.7, max_tokens=50)

prompt = "List three advantages of using vLLM for LLM serving."

# streaming=True вернёт итератор по токенам
for output in llm.generate(prompt, sampling_params, streaming=True):
    # output.outputs[0].text содержит накопленный текст
    print(output.outputs[0].text, end="", flush=True)

print()


В vLLM можно запускать в режиме сервера странным способом - через выполнение Python модуля<br>


| Модуль                                   | Назначение                      |
|-------------------------------------------|----------------------------------|
| `vllm.entrypoints.openai.api_server`      | OpenAI-совместимый API сервер    |
| `vllm.entrypoints.api_server`             | Нативный vLLM API сервер         |
| `vllm.entrypoints.llm`                    | CLI генератор                    |
| `vllm.entrypoints.chat`                   | Чат в терминале                  |
| `vllm.entrypoints.tokenizer`              | Отладка токенизатора             |
| `vllm.entrypoints.convert_lora_weights`   | Конвертация LoRA                 |
| `vllm.entrypoints.slurm.*`                | Запуск в SLURM                   |
| `vllm.entrypoints.megablocks`             | Конверсия в megablocks           |


In [ ]:
python -m vllm.entrypoints.openai.api_server \
  --model meta-llama/Llama-3-8B-Instruct \
  --host 0.0.0.0 \
  --port 8000


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"  # vLLM игнорирует, но поле нужно
)

response = client.chat.completions.create(
    model="meta-llama/Llama-3-8B-Instruct",
    messages=[
        {"role": "user", "content": "Explain what vLLM is in 3 bullet points."}
    ],
    max_tokens=100,
    temperature=0.7,
)

print(response.choices[0].message.content)


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
)

stream = client.chat.completions.create(
    model="meta-llama/Llama-3-8B-Instruct",
    messages=[{"role": "user", "content": "Give me a bullet list of 5 serving engines."}],
    max_tokens=128,
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="", flush=True)

print()


In [ ]:
import requests
import json

url = "http://localhost:8000/v1/chat/completions"

payload = {
    "model": "meta-llama/Llama-3-8B-Instruct",
    "messages": [
        {"role": "user", "content": "Summarize the concept of PagedAttention."}
    ],
    "max_tokens": 100,
    "temperature": 0.7,
}

headers = {
    "Content-Type": "application/json",
    "Authorization": "Bearer EMPTY"
}

resp = requests.post(url, data=json.dumps(payload), headers=headers)
print(resp.json()["choices"][0]["message"]["content"])


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
)

resp = client.embeddings.create(
    model="some-embedding-model-or-llm",
    input=[
        "First sentence to embed.",
        "Second sentence to embed.",
    ],
)

for i, emb in enumerate(resp.data):
    print(f"Vector {i} length:", len(emb.embedding))


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
)

resp = client.chat.completions.create(
    model="meta-llama/Llama-3-8B-Instruct",
    messages=[{"role": "user", "content": "Give me 3 random animal names."}],
    max_tokens=32,
    temperature=1.2,   # более "безумный"
    top_p=0.85,
    presence_penalty=0.0,
    frequency_penalty=0.2,
)

print(resp.choices[0].message.content)


In [ ]:
# Модель 1
python -m vllm.entrypoints.openai.api_server \
  --model meta-llama/Llama-3-8B-Instruct \
  --port 8000 &

# Модель 2
python -m vllm.entrypoints.openai.api_server \
  --model mistralai/Mistral-7B-Instruct-v0.2 \
  --port 8001 &

llama_client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
mistral_client = OpenAI(base_url="http://localhost:8001/v1", api_key="EMPTY")


In [ ]:
from fastapi import FastAPI
from vllm import LLM, SamplingParams

app = FastAPI()
llm = LLM(model="meta-llama/Llama-3-8B-Instruct")

@app.post("/generate")
async def generate(payload: dict):
    prompt = payload["prompt"]
    params = SamplingParams(
        temperature=payload.get("temperature", 0.7),
        max_tokens=payload.get("max_tokens", 64),
    )
    outputs = llm.generate(prompt, params)
    return {"text": outputs[0].outputs[0].text}


In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model="meta-llama/Llama-3-8B-Instruct")

sampling_params = SamplingParams(
    n=3,               # 3 варианта ответа
    temperature=0.9,
    top_k=40,
    top_p=0.9,
    max_tokens=64,
    stop=["\n\n"],     # остановка по токенам/строкам
)

prompt = "Give me 3 short taglines for a serving engine library."

outputs = llm.generate(prompt, sampling_params)

for i, o in enumerate(outputs[0].outputs):
    print(f"=== Variant {i+1} ===")
    print(o.text)
